# Run calibration

Outcomes:
- Prepare ACS demographic targets and ATUS conditional cluster distribution table
- Assign mobility users to ATUS behavioral clusters via k-NN on the precomputed distance matrix
- Run two-stage calibration (IPF demographic + behavioral raking) with replicate weights
- Export main and replicate weights

**Pipeline position:** follows `walkthrough_02_process_acs.ipynb` and `walkthrough_03_process_atus.ipynb`

In [1]:
import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2
from helpers import prep_calibration_inputs

from mobcalibrate import Calibrator

In [2]:
# ======================
# PATHS
# ======================
DATA_DIR = "data/processed"
CBSA_CODE = 38060

# Inputs from previous steps
DIST_MATRIX_FILE = f"{DATA_DIR}/distance_matrix_3cat_full_embedding.parquet"
ATUS_META_FILE   = f"{DATA_DIR}/atus_meta_{CBSA_CODE}.csv"
ACS_CBG_FILE     = f"{DATA_DIR}/acs_cbg_distr_{CBSA_CODE}.csv"
INCOME_MARGIN_FILE = f"{DATA_DIR}/acs_income_margins_{CBSA_CODE}.csv"
AGE_MARGIN_FILE    = f"{DATA_DIR}/acs_age_margins_{CBSA_CODE}.csv"
MEDOID_INFO_FILE = f"{DATA_DIR}/atus_medoid_info_{CBSA_CODE}.json"

# Outputs
OUT_WEIGHTS_MAIN = f"{DATA_DIR}/weights_main_{CBSA_CODE}.csv"
OUT_WEIGHTS_REPLICATES = f"{DATA_DIR}/weights_replicates_{CBSA_CODE}.npy"

# ======================
# STRATIFICATION VARIABLES
# (must match the coding used in walkthrough_03)
# ======================
ROW_VAR          = "income"       # row stratification variable (matches atus_meta column)
COL_VAR          = "age"          # col stratification variable (matches atus_meta column)
CLUSTER_LABEL_COL = "cluster_label"
WEIGHT_COL        = "TUFINLWGT"
GEOID_COL         = "GEOID"

NUM_CLUSTERS = 4

# ======================
# KNN ASSIGNMENT
# ======================
KNN_K          = 10    # number of nearest ATUS neighbors
KNN_THRESHOLD  = 0.5   # min fraction of neighbors required to agree on a label

# ======================
# CALIBRATOR SETTINGS
# ======================
NUM_REPLICATES = 160
SEED           = 1234

## 1. Load data

In [3]:
atus_meta  = pd.read_csv(ATUS_META_FILE)
dist       = pd.read_parquet(DIST_MATRIX_FILE)
acs_cbg    = pd.read_csv(ACS_CBG_FILE, dtype={GEOID_COL: str})
row_margin = pd.read_csv(INCOME_MARGIN_FILE)
col_margin = pd.read_csv(AGE_MARGIN_FILE)
medoid_info = prep_calibration_inputs.load_medoid_info(MEDOID_INFO_FILE)

print(f"ATUS respondents: {len(atus_meta)}")
print(f"Mobility users:   {len(dist)}")
print(f"CBGs in ACS:      {len(acs_cbg)}")

ATUS respondents: 2033
Mobility users:   141530
CBGs in ACS:      2987


In [7]:
# hypothetical user ids (since real ones cannot be published due to data agreement)
dist['user_id'] = range(1, len(dist)+1)

# fix bug in distance matrix
dist['20190504191857'] = dist.loc[:, '20190504191857'].str.replace('\x18', '').astype(float)

## 2. Prepare ACS targets

`prep_acs_targets` normalizes the marginal counts into probability distributions
and extracts category label arrays. These are passed directly to the `Calibrator`.

In [4]:
acs_targets = prep_calibration_inputs.prep_acs_targets(
    row_margin_df = row_margin,
    col_margin_df = col_margin,
    row_var = "hh_income",
    col_var = "age_group",
)

print("Row categories (income):", acs_targets["row_cats"])
print("Col categories (age):   ", acs_targets["col_cats"])
print("Target population:      ", acs_targets["target_pop_tot"])

Row categories (income): ['<35k' '35k-75k' '75k-125k' '125k+']
Col categories (age):    ['18-24' '25-44' '45-66' '67+']
Target population:       3708148


## 3. Prepare ATUS behavioral target table

Computes P(cluster | demographic stratum) from ATUS respondent weights.
Rows index joint demographic strata (income × age), columns index clusters.
Each row sums to 1.

In [5]:
atus_target_P = prep_calibration_inputs.prep_atus_target(
    atus_meta_df      = atus_meta,
    row_var           = ROW_VAR,
    col_var           = COL_VAR,
    cluster_label_col = CLUSTER_LABEL_COL,
    weight_col        = WEIGHT_COL,
    num_row_cats      = acs_targets["num_row_cats"],
    num_col_cats      = acs_targets["num_col_cats"],
    num_clusters      = NUM_CLUSTERS,
)

print(f"Target table shape: {atus_target_P.shape}  (strata * clusters)")
atus_target_P

Target table shape: (16, 4)  (strata * clusters)


array([[0.42763161, 0.03429583, 0.23827778, 0.29979478],
       [0.36993845, 0.11836833, 0.31559443, 0.19609879],
       [0.36191645, 0.22859029, 0.21176841, 0.19772486],
       [0.40904004, 0.32358656, 0.04104545, 0.22632795],
       [0.378268  , 0.06594847, 0.25098553, 0.304798  ],
       [0.20770882, 0.07507449, 0.44093081, 0.27628587],
       [0.3473284 , 0.1097163 , 0.30674674, 0.23620855],
       [0.39891894, 0.22335438, 0.02516295, 0.35256373],
       [0.36988238, 0.20372292, 0.26291417, 0.16348054],
       [0.22803558, 0.06123894, 0.4485094 , 0.26221608],
       [0.30067875, 0.0745933 , 0.3461343 , 0.27859365],
       [0.63952756, 0.13966989, 0.00989712, 0.21090544],
       [0.37892297, 0.        , 0.23703972, 0.38403731],
       [0.1876211 , 0.06620544, 0.50547551, 0.24069795],
       [0.25635734, 0.10610764, 0.39806598, 0.23946903],
       [0.44594546, 0.20819012, 0.07414072, 0.2717237 ]])

## 4. Assign mobility users to ATUS clusters

Uses k-NN voting on the precomputed distance matrix. Each mobility user is
assigned the plurality cluster among their `KNN_K` nearest ATUS neighbors,
provided that cluster accounts for at least `KNN_THRESHOLD` of those neighbors.
Users below the threshold, or farther than the medoid distance threshold for
their cluster, are marked unassigned (`-1`) and excluded from the behavioral
raking step (their demographic weights are still computed in stage 1).

In [8]:
assigned_labels = prep_calibration_inputs.assign_mobility_clusters(
    dist_df           = dist,
    atus_meta_df      = atus_meta,
    cluster_label_col = CLUSTER_LABEL_COL,
    k                 = KNN_K,
    threshold         = KNN_THRESHOLD,
    medoid_indices    = medoid_info['medoid_indices'],
    medoid_thresholds = medoid_info['medoid_thresholds'],
)

n_assigned   = int((assigned_labels >= 0).sum())
n_unassigned = int((assigned_labels == -1).sum())
print(f"Assigned:   {n_assigned}  ({n_assigned / len(assigned_labels):.1%})")
print(f"Unassigned: {n_unassigned}  ({n_unassigned / len(assigned_labels):.1%})")
#pd.Series(assigned_labels).value_counts().sort_index().rename("count")

100%|██████████| 141530/141530 [00:01<00:00, 91729.15it/s]

Assigned:   133830  (94.6%)
Unassigned: 7700  (5.4%)


## 5. Filter to users with valid home CBGs

Drops mobility users whose home GEOID does not appear in the ACS CBG table.
These users cannot be calibrated because no demographic distribution is available
for their home census block group.

In [9]:
user_ids, home_cbgs, assigned_labels, n_dropped = prep_calibration_inputs.filter_valid_users(
    users_df        = dist,
    acs_cbg_df      = acs_cbg,
    assigned_labels = assigned_labels,
    geoid_col       = GEOID_COL,
)

print(f"Users retained: {len(home_cbgs)}")
print(f"Users dropped (missing CBG): {n_dropped}")
print("Proportion of assigned cluster labels:")
pd.Series(assigned_labels).value_counts(normalize=True).sort_index().rename("proportion")

35151 rows have user GEOID missing from ACS GEOID (or null), e.g. ['040136103001' '040130506062' '040138171001' '040130405172'
 '040138118002']
Unique missing GEOIDs (excluding null): 504
Users retained: 106379
Users dropped (missing CBG): 35151
Proportion of assigned cluster labels:


-1    0.054766
 0    0.213322
 1    0.187979
 2    0.319556
 3    0.224377
Name: proportion, dtype: float64

In [10]:
print(user_ids[:5])
print(home_cbgs[:5])
print(assigned_labels[:5])

0    1
1    4
2    5
3    6
4    8
Name: user_id, dtype: int64
['040131112011' '040130925004' '040130715064' '040132168511'
 '040131167031']
[0 1 3 0 2]


## 6. Run calibration

Initializes the `Calibrator` and runs the two-stage procedure for the main weight
set and all replicates. Each replicate independently samples demographic codes from
the CBG-level ACS distributions before raking, propagating individual-level
demographic uncertainty into the weight variance.

`create_main_weights()` and `create_replicate_weights()` both use the same `mode`
argument, which controls which calibration stages are run. The default `"behavioural_full"`
runs both stage 1 (demographic IPF) and stage 2 (behavioral raking).

In [11]:
calibrator = Calibrator(
    unit_ids               = user_ids,
    home_cbgs              = home_cbgs,
    assigned_cluster_labels = assigned_labels,
    acs_cbg_probs_df       = acs_cbg,
    acs_row_var_name       = ROW_VAR,
    acs_col_var_name       = COL_VAR,
    acs_row_cats           = acs_targets["row_cats"],
    acs_col_cats           = acs_targets["col_cats"],
    acs_row_margin         = acs_targets["row_margin"],
    acs_col_margin         = acs_targets["col_margin"],
    atus_target_table      = atus_target_P,
    target_pop_tot         = acs_targets["target_pop_tot"],
    geoid_col              = GEOID_COL,
    num_replicates         = NUM_REPLICATES,
    seed                   = SEED,
)

calibrator.create_main_weights()
calibrator.create_replicate_weights()
print("Weights created. Use get_main_weights() or get_replicate_weights() to obtain weights.")

Weights created. Use get_main_weights() or get_replicate_weights() to obtain weights.


## 7. Inspect weights

In [12]:
# Main weights as a DataFrame
# Columns: weight_behavioural_full, sampled_<row>_code, sampled_<col>_code, sampled_joint_stratum_code
main_weights = calibrator.get_main_weights()
main_weights

,unit_id,weight_behavioural_full,sampled_income_code,sampled_age_code,sampled_joint_stratum_code,assigned_cluster_label
0,1,38,1,1,5,0
1,4,20,1,2,6,1
2,5,29,2,3,11,3
3,6,63,3,3,15,0
4,8,33,2,2,10,2
...,...,...,...,...,...,...
106374,141526,0,0,1,1,-1
106375,141527,63,3,3,15,0
106376,141528,12,3,1,13,1
106377,141529,25,0,1,1,1


In [13]:
# Replicate weights as a 3D numpy array: (num_replicates, N, num_cols)
rep_weights = calibrator.get_replicate_weights(return_df=False)
print(f"Replicate weights shape: {rep_weights.shape}")

# Access replicate r (0-indexed) via rep_weights[r]
# e.g. first replicate:
rep_weights[0]

# NOTE: 
## when weights are returned as np array, 
## unit_ids are not returned along with it due to 
## the possibility of them being a different dtype

Replicate weights shape: (160, 106379, 6)


array([[38,  0, 63,  1,  3,  7],
       [33,  0, 23,  2,  3, 11],
       [42,  0, 45,  0,  3,  3],
       ...,
       [31,  0, 38,  3,  3, 15],
       [38,  0, 15,  1,  1,  5],
       [33,  0,  0,  2,  3, 11]])

## 8. Export

In [ ]:
#main_weights.to_csv(OUT_WEIGHTS_MAIN, index=False)
#np.save(OUT_WEIGHTS_REPLICATES, calibrator.get_replicate_weights(return_df=False))

#print(f"Main weights saved to:      {OUT_WEIGHTS_MAIN}")
#print(f"Replicate weights saved to: {OUT_WEIGHTS_REPLICATES}")